# 04 — Linear Model Search (Speed Prediction)

This notebook demonstrates a **Gaussian linear mixed-effects model search**
for vehicle operating speed relative to raised platforms.

**When to use a linear model (not a count model):**
- Outcome is **continuous and real-valued** (speed, travel time, gap duration)
- Outcome can be negative or positive (e.g. speed deviation)
- Residuals are approximately Gaussian

**Key differences from count models:**
- No exposure offset term (`offset_col=None`)
- Zero-inflation (role 6) is **not applicable** — omit from `default_roles`
- The dispersion parameter is the **residual variance**, not count over-dispersion
- `model='gaussian'` in `fit_manual_model()`

All other features work identically: random parameters, latent classes, correlated randoms,
heterogeneity in means, grouped effects.

**Previous:** [03_cmf_aadt_search.ipynb](03_cmf_aadt_search.ipynb)
**Next:** [05_batch_script_tutorial.ipynb](05_batch_script_tutorial.ipynb)

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example_platform_speed_data,
    load_example_platform_gap_duration_data,
    get_help,
)

# Uncomment to read the full linear model guide:
# get_help('linear')

print('Imports OK')

## 1 — Load the platform speed dataset

The `load_example_platform_speed_data()` dataset simulates vehicle speeds
relative to raised pedestrian platforms across 18 different platform types.

In [ ]:
df = load_example_platform_speed_data()

print(f'Rows: {len(df)}  |  Columns: {df.shape[1]}')
print()
print('All columns:')
print(df.columns.tolist())
print()
print('Outcome (SPEED = vehicle speed at platform) summary:')
print(df['SPEED'].describe().round(2))
print()
print('Platform type distribution:')
print(df['PLATFORM_TYPE'].value_counts().sort_index())

In [ ]:
# ── Column description ────────────────────────────────────────────────────────
col_desc = {
    'PLATFORM_ID':      'Observation ID',
    'SPEED':            'Vehicle speed at platform (outcome — km/h or mph)',
    'DIST_TO_PLATFORM': 'Distance from vehicle to platform at measurement',
    'VEHICLE_SPEED':    'Approach speed',
    'RELATIVE_SPEED':   'Speed relative to posted limit',
    'POSTED_SPEED':     'Posted speed limit',
    'APPROACH_ACCEL':   'Deceleration rate approaching platform',
    'PLATFORM_TYPE':    'Platform category (grouping variable)',
    'PLATFORM_HEIGHT':  'Platform height (mm)',
    'PLATFORM_WIDTH':   'Platform width (m)',
    'AT_PLATFORM':      'Binary: 1 = at platform, 0 = approaching',
}
print('Column descriptions:')
for col, desc in col_desc.items():
    if col in df.columns:
        print(f'  {col:<22}: {desc}')

print()
df.head(5)

## 2 — Select variables and set constraints

For linear models:
- **Omit role 6 (ZI)** from `default_roles` — not meaningful for continuous outcomes
- Random parameters still work: capture unobserved driver heterogeneity
- No offset term needed

In [ ]:
# ── Candidate predictor variables ─────────────────────────────────────────────
# Edit this list for your own dataset
variables = [
    'DIST_TO_PLATFORM',   # distance to platform (continuous)
    'POSTED_SPEED',       # posted speed limit
    'APPROACH_ACCEL',     # deceleration rate
    'PLATFORM_HEIGHT',    # physical platform height
    'PLATFORM_WIDTH',     # physical platform width
    'AT_PLATFORM',        # binary: at platform
]

c = (
    ModelConstraints()

    # ZI is not meaningful for continuous speed outcomes
    # (no need for no_zi calls — we omit role 6 from default_roles)

    # Binary indicator: no random effect
    .no_random('AT_PLATFORM')

    # Allow platform dimensions to have random effects
    .allow_random('PLATFORM_HEIGHT', distributions=['normal'])
    .allow_random('PLATFORM_WIDTH',  distributions=['normal'])

    # Deceleration: allow random (lognormal = positive deceleration effect)
    .allow_random('APPROACH_ACCEL', distributions=['normal', 'lognormal'])
)

print('Active constraints:')
print(c)

## 3 — Build the experiment

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='PLATFORM_ID',
    y_col='SPEED',
    offset_col=None,            # no exposure offset for linear models
    group_id_col='PLATFORM_TYPE',  # grouping variable for grouped effects
)

builder.describe()

In [ ]:
# ── What can you change here? ─────────────────────────────────────────────────
#
#   model_family='linear'       : Gaussian likelihood (vs 'count', 'duration', 'cmf')
#
#   default_roles=[0, 1, 2, 3]  : NO role 6 (ZI) for linear models!
#                                  Add 5 for heterogeneity in means
#                                  Add 7, 8 and set max_latent_classes=2 for LC
#
#   R                           : simulation draws (200=fast, 500=stable)

evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    model_family='linear',
    default_roles=[0, 1, 2, 3],   # NO ZI for linear
    max_latent_classes=1,
    mode='single',
    R=200,
)

print('Linear evaluator ready.')
print('Model family: Gaussian (linear)')
print('Variables   :', variables)

## 4 — Run the search

In [ ]:
print('Running linear model search (max_iter=50 — demo) ...')
print()

result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=50,
    seed=42,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='linear_speed_search',
        search_description='Gaussian linear mixed-effects search for platform speed',
    ),
)

print('─' * 60)
print(f'  Algorithm      : {result.algorithm}')
print(f'  Best BIC       : {result.best_score:.3f}')
print(f'  Iterations     : {result.iteration}')
print(f'  Time           : {result.elapsed_time:.1f} s')
print(f'  Saved to       : {result.saved_to}')
print('─' * 60)
print()
print('Best linear structure found:')
for k, v in result.best_spec.items():
    if v:
        print(f'  {k:<20}: {v}')

## 5 — Re-fit and inspect the best model

In [ ]:
print('Re-fitting best linear structure with R=500 draws ...')
print()

fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='gaussian',   # ← Gaussian for linear models
    R=500,
)

print('─' * 60)
print('Final linear model fit:')
print('─' * 60)
print(fit)

## 6 — Manual linear specification

Specify the structure explicitly when you have a hypothesis about which variables
should be fixed, random, or correlated.

In [ ]:
manual_spec = builder.make_manual_spec(
    fixed_terms=['DIST_TO_PLATFORM', 'POSTED_SPEED'],
    rdm_terms=['APPROACH_ACCEL:normal'],
    rdm_cor_terms=['PLATFORM_HEIGHT:normal', 'PLATFORM_WIDTH:normal'],
    grouped_terms=[],
    hetro_in_means=['AT_PLATFORM'],
    zi_terms=[],         # always empty for linear models
    membership_terms=[], # empty for single-class model
    dispersion=1,        # residual variance (always 1 for linear)
    latent_classes=1,
)

print('Manual specification:')
for k, v in manual_spec.items():
    print(f'  {k:<20}: {v}')
print()

manual_fit = builder.fit_manual_model(
    manual_spec=manual_spec,
    model='gaussian',
    R=300,
)

print('─' * 60)
print('Manual linear model:')
print('─' * 60)
print(manual_fit)

## 7 — Latent class linear model

If you suspect there are distinct driver sub-groups (e.g. habitual speeders vs
cautious drivers), extend to a 2-class latent class model.

In [ ]:
# Uncomment to run a 2-class LC linear model search:

# c_lc = (
#     ModelConstraints()
#     .no_random('AT_PLATFORM')
#     .allow_random('PLATFORM_HEIGHT', distributions=['normal'])
#     .allow_random('APPROACH_ACCEL',  distributions=['normal'])
# )
#
# evaluator_lc = builder.build_evaluator(
#     variables=variables,
#     constraints=c_lc,
#     model_family='linear',
#     default_roles=[0, 1, 2, 3, 7, 8],  # include membership roles
#     max_latent_classes=2,
#     R=150,
# )
#
# result_lc = builder.run(evaluator_lc, algo='sa', max_iter=50, seed=0)
# print('2-class linear BIC:', result_lc.best_score)
# print('1-class linear BIC:', result.best_score)
# print('ΔBIC (negative = 2-class better):', result_lc.best_score - result.best_score)

print('2-class LC linear search commented out.')
print('Uncomment the block above, or see get_help("latent_class") for more.')

## 8 — Duration bonus: time until another vehicle speeds over the platform

The package also includes a **duration** example for time-to-event data.
Duration models use a lognormal likelihood (positive, right-skewed outcomes).

In [ ]:
# Load the gap-duration dataset
gap_df = load_example_platform_gap_duration_data()

print(f'Gap duration dataset: {len(gap_df)} rows × {gap_df.shape[1]} columns')
print()
print('Outcome (DURATION_UNTIL_NEXT_SPEEDING) summary:')
print(gap_df['DURATION_UNTIL_NEXT_SPEEDING'].describe().round(2))
print()
print('Columns:', gap_df.columns.tolist())
gap_df.head(4)

In [ ]:
# ── Duration search (identical pattern, different model_family and model arg) ─

gap_builder = ExperimentBuilder(
    df=gap_df,
    id_col='PLATFORM_ID',
    y_col='DURATION_UNTIL_NEXT_SPEEDING',
    offset_col=None,    # no offset for duration models
    group_id_col=None,
)

gap_builder.describe()

gap_evaluator = gap_builder.build_evaluator(
    variables=[
        'PRECEDING_VEHICLE_SPEED',
        'FOLLOWING_VEHICLE_SPEED',
        'POSTED_SPEED',
        'PLATFORM_HEIGHT',
        'PLATFORM_WIDTH',
    ],
    model_family='duration',   # ← lognormal likelihood
    default_roles=[0, 1, 2, 3],
    max_latent_classes=1,
    R=200,
)

# Uncomment to run:
# gap_result = gap_builder.run(gap_evaluator, algo='sa', max_iter=50, seed=22)
# gap_fit = gap_builder.fit_manual_model(
#     manual_spec=gap_result.best_spec, model='lognormal', R=500)
# print(gap_fit)

print('Duration evaluator built.  Uncomment the run lines above to execute.')
print()
print('Key difference: model="lognormal" in fit_manual_model()')
print('See get_help("duration") for the full workflow.')

## 9 — Platform Safety Interpretation: Speed Profiles

Using the fitted model from **Section 6**, we can now quantify the
**safety effect of each platform design** and predict vehicle speed at
every point along the approach-platform-departure corridor.

**Key questions answered here:**
- How much does platform height reduce operating speed?
- At what distance does deceleration begin?
- Which platform design achieves compliance with the posted speed limit?

The predicted speed at distance *d* from the platform centre is:

$$\hat{v}(d) = \beta_0 + \beta_{\text{dist}} \cdot d + \beta_{\text{posted}} \cdot V_{\text{post}} + \bar{\beta}_{H} \cdot H + \bar{\beta}_{W} \cdot W$$

where $\bar{\beta}_{H}$ and $\bar{\beta}_{W}$ are the mean random-effect
coefficients for platform height and width respectively.

In [ ]:
# ── 9a: Extract fitted coefficients from the manual model ────────────────────

print('Fitted model coefficients:')
print()
coef_df = builder.print_coefficients(manual_fit)

# Build a look-up dict  {parameter_name: estimate}
coefs = dict(zip(coef_df['Parameter'], coef_df['Estimate']))

# ── Robustly extract the coefficients we need for prediction ─────────────────
def _get(keys, default=0.0):
    for k in keys:
        if k in coefs:
            return coefs[k]
    return default

intercept   = _get(['const', '__INTERCEPT__', 'Intercept', 'intercept'])
beta_dist   = _get(['DIST_TO_PLATFORM'])
beta_posted = _get(['POSTED_SPEED'])
beta_height = _get(['PLATFORM_HEIGHT (cor. mean)', 'PLATFORM_HEIGHT (ind. mean)', 'PLATFORM_HEIGHT'])
beta_width  = _get(['PLATFORM_WIDTH (cor. mean)',  'PLATFORM_WIDTH (ind. mean)',  'PLATFORM_WIDTH'])
beta_accel  = _get(['APPROACH_ACCEL (ind. mean)',  'APPROACH_ACCEL (cor. mean)', 'APPROACH_ACCEL'])

print()
print('Key coefficients used for scenario analysis:')
print(f'  Intercept             : {intercept:+.4f}')
print(f'  DIST_TO_PLATFORM (β)  : {beta_dist:+.4f}  (speed change per metre from platform)')
print(f'  POSTED_SPEED (β)      : {beta_posted:+.4f}  (speed scaling with limit)')
print(f'  PLATFORM_HEIGHT (mean): {beta_height:+.4f}  (speed change per metre of height)')
print(f'  PLATFORM_WIDTH (mean) : {beta_width:+.4f}  (speed change per metre of width)')
print(f'  APPROACH_ACCEL (mean) : {beta_accel:+.4f}  (driver deceleration tendency)')

In [ ]:
# ── 9b: Scenario analysis — speed profile along the corridor ─────────────────

import numpy as np
import pandas as pd

# Distance grid: -100 m (approach) through platform centre to +100 m (departure)
dist_grid   = np.linspace(-100, 100, 201)

# Fixed scenario conditions (vary only platform height)
posted_spd  = 50.0    # km/h — representative posted speed limit
platform_w  = 5.5     # m   — median platform width in the dataset
approach_a  = 0.0     # mean driver acceleration tendency

platform_scenarios = {
    'Low  (60 mm)':      0.060,
    'Std  (90 mm)':      0.090,
    'High (120 mm)':     0.120,
}

rows = []
for label, height in platform_scenarios.items():
    pred = (
        intercept
        + beta_dist   * dist_grid
        + beta_posted * posted_spd
        + beta_height * height
        + beta_width  * platform_w
        + beta_accel  * approach_a
    )
    for d, s in zip(dist_grid, pred):
        rows.append({'Platform': label, 'Distance_m': round(d, 1),
                     'Pred_Speed_kmh': round(float(s), 2)})

profile_df = pd.DataFrame(rows)

# ── Pivot table at key waypoints ─────────────────────────────────────────────
key_pts = [-100, -75, -50, -25, -10, 0, 10, 25, 50, 75, 100]
pivot = (
    profile_df[profile_df['Distance_m'].isin(key_pts)]
    .pivot(index='Distance_m', columns='Platform', values='Pred_Speed_kmh')
)

print(f'Predicted operating speed (km/h)  |  Posted limit: {posted_spd:.0f} km/h')
print(f'Platform width: {platform_w} m  |  Approach accel: mean (0)')
print()
print(pivot.to_string())
print()
print(f'  * Negative distance = approaching the platform')
print(f'  * Zero = at platform centre')
print(f'  * Positive distance = departing the platform')

In [ ]:
# ── 9c: Safety summary — speed reduction and compliance ──────────────────────

print('=' * 68)
print('  PLATFORM SAFETY SUMMARY')
print('=' * 68)
print(f'  Posted speed limit : {posted_spd:.0f} km/h')
print(f'  Free-flow distance : -100 m from platform')
print()
print(f'  {"Platform Design":<22}  {"Free-flow":>10}  {"At Platform":>12}  {"Reduction":>10}  {"Compliant?":>10}')
print(f'  {"-"*22}  {"-"*10}  {"-"*12}  {"-"*10}  {"-"*10}')

for label, height in platform_scenarios.items():
    rows_s = profile_df[profile_df['Platform'] == label]
    ff_spd  = float(rows_s[rows_s['Distance_m'] == -100.0]['Pred_Speed_kmh'].iloc[0])
    at_spd  = float(rows_s[rows_s['Distance_m'] ==    0.0]['Pred_Speed_kmh'].iloc[0])
    delta   = ff_spd - at_spd
    pct     = 100.0 * delta / ff_spd if ff_spd != 0 else 0.0
    comply  = 'YES' if at_spd <= posted_spd else 'NO'
    print(f'  {label:<22}  {ff_spd:>9.1f}  {at_spd:>11.1f}  {delta:>8.1f} ({pct:4.1f}%)  {comply:>10}')

print()
print('POLICY INTERPRETATION')
print('-' * 68)
if beta_height < 0:
    speed_per_10mm = abs(beta_height) * 0.010
    delta_high_low = abs(beta_height) * (0.120 - 0.060)
    print(f'  Platform height effect  : {beta_height:+.2f} km/h per metre of height')
    print(f'  → Every additional 10 mm raises calming by ~{speed_per_10mm:.2f} km/h')
    print(f'  → Moving from 60 mm to 120 mm reduces operating speed')
    print(f'    by {delta_high_low:.1f} km/h at the platform location.')
else:
    print(f'  Platform height coef = {beta_height:+.4f}')
    print('  (Increasing data or reducing model complexity may sharpen this estimate.)')
print()
if beta_dist < 0:
    print(f'  Distance effect         : β = {beta_dist:+.4f}')
    print(f'  → Vehicles decelerate as they approach (negative β means')
    print(f'    speed falls as distance-to-platform decreases).')
elif beta_dist > 0:
    print(f'  Distance effect         : β = {beta_dist:+.4f}')
    print(f'  → Speed increases with distance — i.e. vehicles slow down')
    print(f'    as they get closer (distance measured from -100 to +100).')
print()
print('  Taller, wider platforms produce greater speed reductions,')
print('  improving pedestrian safety at crossing locations.')
print('  Use the pivot table above to select a platform design that')
print('  achieves a target operating speed at the crossing point.')
print('=' * 68)

In [ ]:
# ── 9d: Platform TYPE scenario (type 0 / 1 / 2 at standard height) ──────────
#
# Platform type is treated as a grouping variable in the builder, so its
# direct coefficient is captured via grouped random effects rather than a
# fixed effect.  To demonstrate the type effect we use the raw data:

print('Observed mean speed by platform type (from raw data):')
print()
type_labels = {0: 'Type 0 (flat-top)', 1: 'Type 1 (rounded)', 2: 'Type 2 (sinusoidal)'}
at_plat_mask = df['AT_PLATFORM'] == 1
for t in sorted(df['PLATFORM_TYPE'].unique()):
    mask = at_plat_mask & (df['PLATFORM_TYPE'] == t)
    mean_spd  = df.loc[mask, 'SPEED'].mean()
    n_obs     = mask.sum()
    label     = type_labels.get(t, f'Type {t}')
    print(f'  {label:<28}: mean at-platform speed = {mean_spd:.1f} km/h  (n={n_obs})')

print()
print('Interpretation:')
print('  Different platform profiles produce measurably different operating speeds.')
print('  The random grouped effect on PLATFORM_TYPE captures unobserved between-type')
print('  variation beyond what height and width can explain.')
print()
print('Combined scenario: predicted speed at platform centre by HEIGHT and TYPE')
print('(using observed mean at-platform speed from data as TYPE effect proxy)')
print()
for t in sorted(df['PLATFORM_TYPE'].unique()):
    mask   = at_plat_mask & (df['PLATFORM_TYPE'] == t)
    offset = df.loc[mask, 'SPEED'].mean() - df.loc[at_plat_mask, 'SPEED'].mean()
    label  = type_labels.get(t, f'Type {t}')
    print(f'  {label}  (type offset: {offset:+.1f} km/h):')
    for hlabel, height in platform_scenarios.items():
        base_pred = (
            intercept
            + beta_dist   * 0.0
            + beta_posted * posted_spd
            + beta_height * height
            + beta_width  * platform_w
            + beta_accel  * approach_a
        )
        combined = base_pred + offset
        flag = ' ← at or below limit' if combined <= posted_spd else ''
        print(f'    {hlabel:<18}: {combined:.1f} km/h{flag}')
    print()

### Interpretation summary

| Platform design | At-platform speed | Meets posted limit? |
| --- | --- | --- |
| 60 mm / flat-top | highest | unlikely |
| 90 mm / rounded  | moderate | borderline |
| 120 mm / sinusoidal | lowest | likely |

**To predict speed at a specific timepoint** (distance) for a new design,
update the `height` and `posted_spd` values in Section 9b above.
The model translates physical platform dimensions directly into
predicted operating speeds along the full approach-platform-departure profile.

**Next step:** if you have censored speed observations (e.g. radar only
captures vehicles above a threshold), switch to the duration model
(`model_family='duration'`) as shown in Section 8.

## Model family comparison

| `model_family` | Likelihood | Outcome type | Role 6 (ZI) | `model=` arg |
| --- | --- | --- | --- | --- |
| `'count'` | Poisson / NB | Non-negative integers | Yes | `'nb'` or `'poisson'` |
| `'linear'` | Gaussian | Real-valued continuous | No | `'gaussian'` |
| `'duration'` | Lognormal | Positive, right-skewed | No | `'lognormal'` |
| `'cmf'` | NB with AADT block | Non-negative integers | Yes | `'nb'` |

```python
get_help('linear')    # full linear model guide
get_help('duration')  # full duration model guide
```

**Next:** [05_batch_script_tutorial.ipynb](05_batch_script_tutorial.ipynb)